In [1]:
import requests
import json
import cdsapi
import xarray as xr
from collections import defaultdict
from zoneinfo import ZoneInfo
import pandas as pd
import numpy as np

In [2]:
url = "https://kukau.org/avg_air_temp_solar_rain.json"
resp = requests.get(url)
data = resp.json()
stations = defaultdict(list)
for row in data:
    stations[row["station_no"]].append(row)


In [3]:
today = pd.Timestamp.now(tz=ZoneInfo("HST"))
today_utc = today.strftime("%Y-%m-%d")
date_str = f"{today_utc}/{today_utc}"
date_str

'2025-11-25/2025-11-25'

In [4]:
station = "PW74561"

total_rain_in_last3 = sum(d["rain_in"] for d in data if d["station_no"] == station)


In [5]:
dataset = "cams-global-atmospheric-composition-forecasts"
request = {
    "variable": [
        "2m_temperature",
        "surface_solar_radiation_downwards",
        "total_precipitation"
    ],
    "date": [date_str],
    "time": ["00:00"],
    "leadtime_hour": [
        "0",
        "1",
        "2",
        "3",
        "4",
        "5",
        "6",
        "7",
        "8",
        "9",
        "10",
        "11",
        "12",
        "13",
        "14",
        "15",
        "16",
        "17",
        "18",
        "19",
        "20",
        "21",
        "22",
        "23",
        "24",
        "25",
        "26",
        "27",
        "28",
        "29",
        "30",
        "31",
        "32",
        "33",
        "34",
        "35",
        "36",
        "37",
        "38",
        "39",
        "40",
        "41",
        "42",
        "43",
        "44",
        "45",
        "46",
        "47",
        "48",
        "49",
        "50",
        "51",
        "52",
        "53",
        "54",
        "55",
        "56",
        "57",
        "58",
        "59",
        "60",
        "61",
        "62",
        "63",
        "64",
        "65",
        "66",
        "67",
        "68",
        "69",
        "70",
        "71",
        "72",
        "73",
        "74",
        "75",
        "76",
        "77",
        "78",
        "79",
        "80",
        "81",
        "82",
        "83",
        "84",
        "85",
        "86",
        "87",
        "88",
        "89",
        "90",
        "91",
        "92",
        "93",
        "94",
        "95",
        "96",
        "97",
        "98",
        "99",
        "100",
        "101",
        "102",
        "103",
        "104",
        "105",
        "106",
        "107",
        "108",
        "109",
        "110"
    ],
    "type": ["forecast"],
    "data_format": "grib",
    "area": [8, 134, 7, 135]
}

client = cdsapi.Client()
result = client.retrieve(dataset, request)
grib_path = result.download()  


2025-11-25 17:00:30,499 INFO Request ID is 927d793f-ec33-439d-b84f-6a6ce9bdabb3
2025-11-25 17:00:30,914 INFO status has been updated to accepted
2025-11-25 17:00:38,465 INFO status has been updated to running
2025-11-25 17:01:29,195 INFO status has been updated to successful


f9be27513e68ceda72da2ce33311ea92.grib:   0%|          | 0.00/41.0k [00:00<?, ?B/s]

In [6]:
ds = xr.open_dataset(grib_path, engine="cfgrib")
ds = ds.assign_coords(valid_time = ds.time + ds.step)

palau = ds.sel(latitude=7.5150, longitude=134.5825, method="nearest")

valid_local = (
    pd.to_datetime(palau.valid_time.values)
      .tz_localize("UTC")
      .tz_convert("Pacific/Palau")
      .tz_localize(None)
)

palau = palau.assign_coords(valid_time=("step", valid_local))
dates = palau.valid_time.dt.floor("D")

/opt/anaconda3/lib/python3.13/site-packages/xarray/backends/plugins.py:110: RuntimeWarning: Engine 'rasterio' loading failed:
dlopen(/opt/anaconda3/lib/python3.13/site-packages/rasterio/_base.cpython-313-darwin.so, 0x0002): Symbol not found: __ZN3Aws15CognitoIdentity21CognitoIdentityClientC1ERKNSt3__110shared_ptrINS_4Auth22AWSCredentialsProviderEEENS3_INS_8Endpoint20EndpointProviderBaseINS_6Client26GenericClientConfigurationENS9_17BuiltInParametersENS9_23ClientContextParametersEEEEERKSC_
  Referenced from: <C061D09E-B1FB-389D-9495-4BBC60C7995E> /opt/homebrew/Cellar/aws-sdk-cpp/1.11.600/lib/libaws-cpp-sdk-identity-management.dylib
  Expected in:     <7B72F0AE-72AD-3134-B98A-065897A98791> /opt/anaconda3/lib/libaws-cpp-sdk-cognito-identity.dylib
  external_backend_entrypoints = backends_dict_from_pkg(entrypoints_unique)
/opt/anaconda3/lib/python3.13/site-packages/cfgrib/xarray_plugin.py:132: FutureWarning: In a future version of xarray decode_timedelta will default to False rather than No

In [7]:
today = pd.Timestamp.now(tz="Pacific/Palau").date()
today_str = today.strftime("%Y-%m-%d")

today = pd.Timestamp(today_str).date()
next_3_days = [(today + pd.Timedelta(days=i)) for i in range(1, 4)]
next_3_str = [d.strftime("%Y-%m-%d") for d in next_3_days]

In [8]:
#Temp 
t2m = (palau['t2m'] - 273.15) * 9/5 + 32
daily_avg_t = t2m.groupby(dates).max("step")
daily_avg_t_series = daily_avg_t.to_series()

today_t = float(daily_avg_t_series.get(today_str, np.nan))
next_3_t_values = [
    float(daily_avg_t_series.get(day, np.nan))
    for day in next_3_str
]
next_3_t_max = np.nanmax(next_3_t_values)

In [9]:
def accumulated_to_incremental(data_arr):
    vals = data_arr.values
    inc = np.empty_like(vals)
    inc[0] = 0
    inc[1:] = vals[1:] - vals[:-1]
    return xr.DataArray(
        inc,
        coords={"step": data_arr.step},
        dims=["step"],
        name=data_arr.name + "_inc"
    )
        

ssrd_acc = palau["ssrd"]
ssrd_inc = accumulated_to_incremental(ssrd_acc)
    
ssrd_MJ = ssrd_inc / 1e6
ssrd_MJ = ssrd_MJ.assign_coords(valid_time=("step", valid_local))
ssrd_MJ = ssrd_MJ.swap_dims({"step": "valid_time"})
    
ssrd_series = ssrd_MJ.to_pandas()
daily_solar = ssrd_series.resample("D").sum()
    
today_solar = float(daily_solar.get(today_str, np.nan))
        

tp_acc = palau["tp"]                 # m accumulated
tp_inc = accumulated_to_incremental(tp_acc)
    
tp_mm = tp_inc * 1000                # convert m → mm
tp_mm = tp_mm.assign_coords(valid_time=("step", valid_local))
tp_mm = tp_mm.swap_dims({"step": "valid_time"})
    
tp_series = tp_mm.to_pandas()
daily_precip_mm = tp_series.resample("D").sum()
    
today_precip_mm = float(daily_precip_mm.get(today_str, np.nan))
    
    
next_3_solar_values = [
    float(daily_solar.get(day, np.nan)) for day in next_3_str
]
next_3_precip_values = [
    float(daily_precip_mm.get(day, np.nan)) for day in next_3_str
]
    
next_3_solar_max = float(np.nanmax(next_3_solar_values))
next_3_precip_sum = float(np.nansum(next_3_precip_values))


In [13]:
palau

<xarray.Dataset> Size: 3kB
Dimensions:     (step: 111)
Coordinates:
    number      int64 8B 0
    time        datetime64[ns] 8B 2025-11-25
  * step        (step) timedelta64[ns] 888B 00:00:00 ... 4 days 14:00:00
    surface     float64 8B 0.0
    latitude    float64 8B 7.4
    longitude   float64 8B 134.4
    valid_time  (step) datetime64[ns] 888B 2025-11-25T09:00:00 ... 2025-11-29...
Data variables:
    t2m         (step) float32 444B 300.5 300.7 300.9 ... 300.6 300.6 300.9
    ssrd        (step) float32 444B 0.0 2.381e+06 ... 9.662e+07 9.662e+07
    tp          (step) float32 444B 0.0 1.4e-07 0.0 ... 0.01162 0.01182 0.01194
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             European Centre for Medium-Range Weather Forecasts
    history:                 2025-11-25T17:01 GRIB to CDM+CF via cfgrib-0.9.1...

In [10]:
daily_solar

valid_time
2025-11-25    17.631275
2025-11-26    21.323059
2025-11-27    16.667456
2025-11-28    21.227264
2025-11-29    19.766993
Freq: D, Name: ssrd_inc, dtype: float32

In [10]:
if today_t > 87 and today_solar > 20:
    if  next_3_t_max > 87 and next_3_solar_max > 20:
        index = "HOT/HOT"
    elif next_3_precip_sum > 1.5:
        index = "HOT/WET"
    elif total_rain_in_last3 > 0.5:
        index = "WET/HOT"
else:
    index = "None"

In [18]:
import json

data = {
    "Index": index,
    "date": today_str,
    "temp_today": today_t,
    "solar_today": today_solar,
    "temp_next3days": next_3_t_max,
    "rain_last3days": total_rain_in_last3,
    "Solar_rad_next3days": next_3_solar_max
}

# Save to file
with open("index.json", "w") as f:
    json.dump(data, f, indent=2)

print("Saved to data.json")


Saved to data.json


In [31]:
row = {
    "date": data.get("date"),
    "Index": data.get("Index"),
    "temp_today": data.get("temp_today"),
    "solar_today": data.get("solar_today"),
    "rain_last3days": data.get("rain_last3days"),
    "Solar_rad_next3days": data.get("Solar_rad_next3days")
}

csv_path = "history.csv"
import os
# Append or create
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
else:
    df = pd.DataFrame([row])

/var/folders/vl/70ggslts0x98b_vgphfybj140000gn/T/ipykernel_92582/470298123.py:15: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)


In [32]:
df.to_csv(csv_path, index=False)